# Exercise 1: Webscraping

Author: Georg Ahnert

In this exercise, we will first recap some basic Pandas functionality for handeling datasets in Python. Then, we will have a look into webscraping and crawling.

You will first have to install the following dependencies:

In [ ]:
%pip install pandas beautifulsoup4 requests urllib3 --upgrade

## Part1: Pandas Recap

First, let's recap some basic Pandas functionalities. 

In [ ]:
import pandas as pd

### Create a Pandas DataFrame

Call the variable `test_df`. It should have the following structure:

|   | column_A | column_B |
|---|----------|----------|
| 2 | 2 | 3  |
| 3 | 4 | 15 |
| 4 | 6 | 75 |

Next, we will work with the Quality of Government Dataset and perform some basic operations

In [ ]:
# read the data directly online:
df=pd.read_csv("http://www.qogdata.pol.gu.se/data/qog_std_cs_jan22.csv")
df.head()

### How many rows does this dataset have?

### How many columns does this dataset have?

### For the following tasks, select these columns from the dataset: 

"cname", "wdi_pop", "wdi_popgr", "wdi_gdpcapcur", "wdi_gdpcapgr", "wdi_area", "wdi_broadb", "ht_region"

### Rename these columns to: "country", "population","population_growth", "gdp_per_capita", "gdp_growth", "area", "internet", "region"

### Create a _categorical_ column from the region codes
the codes (in order correspond to the regions as follows):
"Eastern Europe", "Latin America", "North Africa & Middle East", "Sub-Saharan Africa", "Western Europe and North America", "East Asia","South-East Asia", "South Asia", "Pacific", "Caribbean"

### Select the five countries with the highest population

### What are the mean values for each attribute?

### Which country has the highest population in the region "South-East Asia"?

### Create a new column "population_density"

## Part 2: Web Scraping

### Intro to HTML
HTML - Hyper Text Markup Language (see also [Wikipedia](https://en.wikipedia.org/wiki/HTML))

HTML elements are defined by tags
```
<b>Bold text</b>
```
Tags have attributes
```
<span class="uni">University of Mannheim</span>
```
Tags could be nested
```
<div id="uni-list" class="sfsdf" attribute1="xasdas" attribute2="asfasd">
 <span class="uni">University of Mannheim</span>
 <span class="uni">RWTH Aachen</span>
 <a class="uni">Hello</a>
</div>
```


Here's a [web page example](https://www.uni-mannheim.de/en/academics/programs/). Here's a video that recaps the most important elements of the language: [YouTube](https://youtu.be/salY_Sm6mv4).

### Querying HTML pages with BeautifulSoup

HTML-elements could be selected by name
```
a
```
By ID
```
#uni-list
```
By class
```
.uni
```
Nested selection
```
#uni-list .uni
```

```
#uni-list span.uni
```

See this nice [introduction to CSS selectors](https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_selectors/Selectors_and_combinators).
See also the [BeautifulSoup Documentation](https://beautiful-soup-4.readthedocs.io/en/latest/)

In [ ]:
from bs4 import BeautifulSoup  # import the BeautifulSoup class from the bs4 package

In [ ]:
# an example HTML document in the form of a string
html_doc = '<html lang="en"><head><title>Test document</title></head><body><div id="uni-list"><div class="sub"><a href="https://www.uni-mannheim.de/" class="uni">University of Mannheim</a></div><div class="sub"><a href="http://rwth-aachen.de" class="uni">RWTH Aachen</a></div></div></body></html>'

# initialize a BeautifulSoup object with the given HTML
soup = BeautifulSoup(html_doc)
type(soup)

In [ ]:
# we can also specify the type of parser we want to use
# they work differently well with non-conforming HTML
soup = BeautifulSoup(html_doc, parser='html.parser')

In [ ]:
soup  # let's see what we got

In [ ]:
print(soup.prettify())  # add indentation

### Accessing Elements in the Soup

In [ ]:
soup.title  # access the first 'title' element in the soup

In [ ]:
soup.title.text  # get the text inside the first 'title' element

In [ ]:
print(soup.div.prettify())  # access the first 'div' element and print it pretty

In [ ]:
soup.a  # access the first anchor element (hyperlink)

But what if we want to get **all** hyperlinks? We can use **CSS selectors** to achieve this.

In [ ]:
result = soup.select('a')  # returns a list of all elements that match the name 'a'
result

In [ ]:
soup.select('span')  # the list can also be empty or have only one element

In [ ]:
second = soup.select('a')[1]  # we can extract entries from the list just as usual
second

### More advanced selectors

### Select all elements that have the ID 'uni-list'

### Select all elements that are of class 'uni'

### Select all elements that have name 'div' and class 'sub'

### Select all 'div' elements that are descendents of an element with id 'uni-list'

### Loading actual web data

In [ ]:
import requests

url = "https://www.uni-mannheim.de/en/academics/before-your-studies/programs/"

page = requests.get(url).text  # get the content at that URL and store the page source

soup = BeautifulSoup(page)  # initialize beautiful soup

In [ ]:
soup  # let's see what we got

Let's try to extract the course programs that Uni Mannheim offers

In [ ]:
courses = soup.select('.uma-ps-result-item')
len(courses)

In [ ]:
print(courses[0].prettify())  # inspect the first element

In [ ]:
result = soup.select('.uma-ps-result-item-title')  # we're only interested in the programs' names
result

In [ ]:
[x.text for x in result]  # only the text, not the tags

### Web Crawling

Another thing we could be interested in is all the testimonials that are published on the course programs' websites:

In [ ]:
result = soup.select('.uma-ps-results a')  # get all the anchor elements

for element in result:
    print(element['href'])  # extract the hyperlinks

In [ ]:
# follow all the links
for element in result:
    url = element['href']

    # convert relative to absolute links
    if not(url.startswith('https:/')):
        url = 'https://www.uni-mannheim.de' + url
    
    # get the linked page
    page = requests.get(url).text
    soup = BeautifulSoup(page, 'html.parser')
    testimonial_tags = soup.select('.testimonial-quote')
    
    # if there's a testimonial, print the first one
    if len(testimonial_tags) > 0:
        testimonial = testimonial_tags[0]
        print(testimonial.text.strip())